# 01 - Loan Recovery Foundations and EDA

## Definition
**Loan recovery** is the process of collecting unpaid loan amounts after delinquency/default.

## Theory
Banks estimate expected recovery to reduce credit losses, optimize collection effort, and improve portfolio quality.

## Mathematical intuition
Expected recovery value per borrower can be approximated as:
\[
\mathbb{E}[Recovery] = Outstanding\_Balance \times P(Recovery)
\]

## Financial intuition
A borrower with high outstanding balance and high delinquency can destroy profitability if recovery fails.

## Business impact
Strong recovery analytics improves cashflow, reduces NPA pressure, and guides legal vs non-legal actions.


## Definition
Loan recovery EDA frames where losses originate and how recoverability differs across borrower profiles.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
Borrowers with high days-past-due and weak collateral often require earlier intervention.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [ ]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


## Banking Context: Why This Matters
- **Banking**: controls provisioning and capital adequacy impact from bad loans.
- **NBFCs**: reduces collection cost per recovered dollar.
- **FinTech lenders**: enables faster early-warning decisions at scale.
- **Microfinance institutions**: helps prioritize high-touch outreach where social + financial risk is high.


In [ ]:
loader = LoanDataLoader(DATA_PATH)
df = loader.load()
print(f"Dataset shape: {df.shape}")
display(df.head())


## Data Quality Assessment
### Real-world meaning
Missing values, invalid ranges, and outliers are not only technical issues, they are operational risk signals.


In [ ]:
quality = loader.quality_report(df)
quality_df = pd.DataFrame([
    {"check": k, "count": v} for k, v in quality.invalid_ranges.items()
])
display(pd.DataFrame([quality.to_dict()]).T)
display(quality_df)


## Exploratory Data Analysis
We inspect borrower, loan, behavior, and target patterns using histograms, KDEs, boxplots, violin plots, and heatmaps.


In [ ]:
fe = FeatureEngineer()
enriched = fe.engineer(df)
eda = LoanEDA(enriched)
outputs = eda.generate_all_plots(FIGURES_DIR)
display(eda.summary_table().head(15))
display(eda.relationship_analysis())
outputs


In [ ]:
from IPython.display import Image, display

for image_path in [
    outputs.target_distribution,
    outputs.correlation_heatmap,
    outputs.numeric_histograms,
    outputs.boxplots,
    outputs.violin_plots,
    outputs.relationship_grid,
]:
    display(Image(filename=str(image_path)))


## Interpretation
- Higher `Days_Past_Due` and `Missed_Payments` often align with weaker recovery states.
- Borrowers with stronger collateral coverage generally look easier to recover.
- Income, debt burden, and repayment behavior jointly shape collection strategy decisions.
